In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm  # PyTorch Image Models
import os
import time
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score
import seaborn as sns
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")


In [ ]:
import os
import torch

# --- KONFIGURASYON ---
# ResNeXt50 Modeli (timm kodu: resnext50_32x4d) -> small degil
MODEL_NAME = 'resnext50_32x4d'

EXPERIMENT_NAME = "ResNeXt50_Baseline_MediumLarge_Run1"

# Hiperparametreler
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 2.5e-4
NUM_CLASSES = 8
IMAGE_SIZE = 224
DROPOUT_RATE = 0.3

# Duzenlilestirme ve egitim kararliligi
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
WARMUP_EPOCHS = 3
EARLY_STOPPING_PATIENCE = 10
USE_CLASS_WEIGHTED_LOSS = True
LOSS_NAME = "cross_entropy"

NUM_WORKERS = 0 if os.name == "nt" else 4

cwd = os.getcwd()
if os.path.isdir(os.path.join(cwd, "data", "prepared-data")):
    PROJECT_ROOT = cwd
elif os.path.isdir(os.path.join(cwd, "..", "data", "prepared-data")):
    PROJECT_ROOT = os.path.abspath(os.path.join(cwd, ".."))
else:
    PROJECT_ROOT = cwd

DATA_DIR = os.path.join(PROJECT_ROOT, "data", "prepared-data")

dropout_tag = f"dropout{int(round(DROPOUT_RATE * 10)):02d}"
ls_tag = f"ls{int(round(LABEL_SMOOTHING * 100)):02d}"
wd_tag = f"wd{str(WEIGHT_DECAY).replace('.', 'p').replace('-', 'm')}"
lr_tag = f"lr{LEARNING_RATE:g}".replace(".", "p")
RUN_TAG = f"{dropout_tag}_{ls_tag}_{wd_tag}_bs{BATCH_SIZE}_ep{EPOCHS}_{lr_tag}"
RUN_NAME = f"{EXPERIMENT_NAME}_{RUN_TAG}"

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "models", "pytorch", RUN_NAME)
os.makedirs(OUTPUT_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, f"best_model_{MODEL_NAME}_{RUN_TAG}.pth")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True

print(f"CWD: {cwd}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Cihaz: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Sayisi: {torch.cuda.device_count()}")
else:
    print("GPU: Kullanilamiyor, egitim CPU ile calisacak")
print(f"Model: {MODEL_NAME}")
print(f"Run Tag: {RUN_TAG}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"Kayit Yeri: {OUTPUT_DIR}")
print(f"Best Model Yolu: {BEST_MODEL_PATH}")
print(f"NUM_WORKERS: {NUM_WORKERS}")
print(f"Weight Decay: {WEIGHT_DECAY}")
print(f"Label Smoothing: {LABEL_SMOOTHING}")
print(f"Warmup Epochs: {WARMUP_EPOCHS}")
print(f"Early Stopping Patience: {EARLY_STOPPING_PATIENCE}")
print(f"Image Size: {IMAGE_SIZE}")
print(f"Use Class Weighted Loss: {USE_CLASS_WEIGHTED_LOSS}")


In [ ]:
# ImageNet Normalize Degerleri
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val': transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'test': transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
}

image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x])
                  for x in ['train', 'val', 'test']}

train_targets = np.array(image_datasets["train"].targets)
class_sample_counts = np.bincount(train_targets, minlength=NUM_CLASSES)
class_weight_values = class_sample_counts.sum() / (NUM_CLASSES * np.clip(class_sample_counts, 1, None))
CLASS_WEIGHTS_TENSOR = torch.tensor(class_weight_values, dtype=torch.float32)

dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE,
                             shuffle=(x=="train"), num_workers=NUM_WORKERS, pin_memory=True)
               for x in ["train", "val", "test"]}

dataset_sizes = {x: len(image_datasets[x]) for x in ["train", "val", "test"]}
class_names = image_datasets["train"].classes

print(f"Siniflar: {class_names}")
print(f"Egitim Verisi: {dataset_sizes['train']}")
print(f"Validasyon Verisi: {dataset_sizes['val']}")
print(f"Sinif sayimlari (train): {class_sample_counts.tolist()}")
print(f"Class weights aktif: {USE_CLASS_WEIGHTED_LOSS}")


In [ ]:
def create_model():
    print(f"Model indiriliyor: {MODEL_NAME}...")
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES, drop_rate=DROPOUT_RATE)
    return model

model = create_model()
model = model.to(DEVICE)

loss_weight = CLASS_WEIGHTS_TENSOR.to(DEVICE) if USE_CLASS_WEIGHTED_LOSS else None
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING, weight=loss_weight)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / float(max(1, WARMUP_EPOCHS))
    progress = (epoch - WARMUP_EPOCHS) / float(max(1, EPOCHS - WARMUP_EPOCHS))
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

print("Model GPU'ya yuklendi ve egitime hazir.")
print(f"AdamW + weight_decay={WEIGHT_DECAY}, label_smoothing={LABEL_SMOOTHING}")


In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs):
    required_globals = ["dataloaders", "dataset_sizes", "OUTPUT_DIR", "DEVICE", "BEST_MODEL_PATH", "EARLY_STOPPING_PATIENCE"]
    missing = [name for name in required_globals if name not in globals()]
    if missing:
        raise RuntimeError(f"Eksik degisken(ler): {missing}. Lutfen once veri ve konfigurasyon hucrelerini calistirin.")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(os.path.dirname(BEST_MODEL_PATH), exist_ok=True)

    since = time.time()
    best_acc = 0.0
    best_val_loss = float("inf")
    epochs_without_improvement = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 10)

        for phase in ["train", "val"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc.item())

            if phase == "val":
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                if epoch_loss < best_val_loss:
                    best_val_loss = epoch_loss
                    epochs_without_improvement = 0
                    torch.save(model.state_dict(), BEST_MODEL_PATH)
                    print(f"En Iyi Model (val loss: {best_val_loss:.4f}) -> {BEST_MODEL_PATH}")
                else:
                    epochs_without_improvement += 1
                    if EARLY_STOPPING_PATIENCE > 0:
                        print(f"Val loss iyilesmedi: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")
                    else:
                        print(f"Val loss iyilesmedi: {epochs_without_improvement} epoch")

        scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]
        history["lr"].append(current_lr)
        print(f"Guncel LR: {current_lr:.8f}")

        if EARLY_STOPPING_PATIENCE > 0 and epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Erken durdurma tetiklendi.")
            break

    time_elapsed = time.time() - since
    print(f"\nEgitim Tamamlandi: {time_elapsed // 60:.0f}dk {time_elapsed % 60:.0f}sn")
    print(f"En Iyi Validasyon Dogrulugu: {best_acc:.4f}")
    print(f"En Dusuk Validasyon Kaybi: {best_val_loss:.4f}")
    model.load_state_dict(torch.load(BEST_MODEL_PATH))
    return model, history


In [ ]:
# --- BASLAT ---
required_vars = ["model", "criterion", "optimizer", "scheduler", "train_model"]
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise RuntimeError(f"Eksik degisken(ler): {missing}. Lutfen once model/egitim hazirlik hucrelerini calistirin.")

num_epochs = int(globals().get("EPOCHS", 30))
model, history = train_model(model, criterion, optimizer, scheduler, num_epochs=num_epochs)


In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history["train_acc"], label="Train Acc")
plt.plot(history["val_acc"], label="Val Acc")
plt.title(f"{MODEL_NAME} Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.title(f"{MODEL_NAME} Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

PLOT_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs", MODEL_NAME, "plots")
os.makedirs(PLOT_OUTPUT_DIR, exist_ok=True)
plot_path = os.path.join(PLOT_OUTPUT_DIR, f"training_graph_{MODEL_NAME}_{RUN_TAG}.png")
plt.savefig(plot_path)
plt.show()
print(f"Grafikler kaydedildi: {plot_path}")


In [ ]:
print("\nTEST SETI DEGERLENDIRMESI")

model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in dataloaders["test"]:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("\nSiniflandirma Raporu:")
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

macro_f1 = f1_score(y_true, y_pred, average="macro")
class_recalls = recall_score(y_true, y_pred, average=None, labels=list(range(len(class_names))))
print(f"Macro F1: {macro_f1:.4f}")
print("Class recall degerleri:")
for class_name, rec in zip(class_names, class_recalls):
    print(f"  {class_name}: {rec:.4f}")

report_text = report + f"\nMacro F1: {macro_f1:.6f}\n" + "Class Recall:\n"
for class_name, rec in zip(class_names, class_recalls):
    report_text += f"{class_name}: {rec:.6f}\n"

os.makedirs(OUTPUT_DIR, exist_ok=True)
primary_report_path = os.path.join(OUTPUT_DIR, f"classification_report_{MODEL_NAME}_{RUN_TAG}.txt")
fallback_report_dir = os.path.join(PROJECT_ROOT, "outputs", MODEL_NAME, "reports")
fallback_report_path = os.path.join(fallback_report_dir, f"classification_report_{MODEL_NAME}_{RUN_TAG}.txt")
report_path = primary_report_path
try:
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report_text)
except (FileNotFoundError, OSError):
    os.makedirs(fallback_report_dir, exist_ok=True)
    report_path = fallback_report_path
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report_text)
    print("Uyari: Uzun yol nedeniyle rapor fallback klasorune kaydedildi.")
print(f"Classification report kaydedildi: {report_path}")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Tahmin Edilen")
plt.ylabel("Gercek")
plt.title(f"Confusion Matrix - {MODEL_NAME}")
PLOT_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs", MODEL_NAME, "plots")
os.makedirs(PLOT_OUTPUT_DIR, exist_ok=True)
plot_path = os.path.join(PLOT_OUTPUT_DIR, f"confusion_matrix_{MODEL_NAME}_{RUN_TAG}.png")
plt.savefig(plot_path)
plt.show()
print(f"Confusion matrix kaydedildi: {plot_path}")


In [ ]:
# --- ABLATION KARSILASTIRMA OZETI ---
ablation_csv_path = os.path.join(PROJECT_ROOT, "outputs", MODEL_NAME, f"ablation_results_{MODEL_NAME}.csv")
os.makedirs(os.path.dirname(ablation_csv_path), exist_ok=True)

force_weighted_sampler = globals().get("FORCE_WEIGHTED_SAMPLER", False)
use_weighted_sampler = globals().get("USE_WEIGHTED_SAMPLER", False)
name_to_idx = {name: i for i, name in enumerate(class_names)}
pair_1_err = np.nan
pair_2_err = np.nan
if "cm" in globals() and "esophagitis" in name_to_idx and "normal-z-line" in name_to_idx:
    i, j = name_to_idx["esophagitis"], name_to_idx["normal-z-line"]
    pair_1_err = int(cm[i, j] + cm[j, i])
if "cm" in globals() and "dyed-lifted-polyps" in name_to_idx and "dyed-resection-margins" in name_to_idx:
    i, j = name_to_idx["dyed-lifted-polyps"], name_to_idx["dyed-resection-margins"]
    pair_2_err = int(cm[i, j] + cm[j, i])

current_row = {
    "run_name": RUN_NAME,
    "run_tag": RUN_TAG,
    "model_name": MODEL_NAME,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dropout_rate": DROPOUT_RATE,
    "label_smoothing": LABEL_SMOOTHING,
    "force_weighted_sampler": force_weighted_sampler,
    "use_weighted_sampler": use_weighted_sampler,
    "use_class_weighted_loss": USE_CLASS_WEIGHTED_LOSS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "best_val_acc": float(max(history["val_acc"])) if "history" in globals() and history.get("val_acc") else np.nan,
    "best_val_loss": float(min(history["val_loss"])) if "history" in globals() and history.get("val_loss") else np.nan,
    "test_macro_f1": float(macro_f1) if "macro_f1" in globals() else np.nan,
    "pair_err_esophagitis_normal_z_line": pair_1_err,
    "pair_err_dyed_lifted_vs_resection": pair_2_err
}

if os.path.exists(ablation_csv_path):
    df_existing = pd.read_csv(ablation_csv_path)
else:
    df_existing = pd.DataFrame()
if not df_existing.empty and "run_name" in df_existing.columns:
    df_existing = df_existing[df_existing["run_name"] != RUN_NAME].copy()
df_updated = pd.concat([df_existing, pd.DataFrame([current_row])], ignore_index=True)
df_updated.to_csv(ablation_csv_path, index=False)
print(f"Ablation CSV guncellendi: {ablation_csv_path}")

df = pd.read_csv(ablation_csv_path).copy()
numeric_cols = ["best_val_acc", "best_val_loss", "test_macro_f1", "pair_err_esophagitis_normal_z_line", "pair_err_dyed_lifted_vs_resection"]
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
df["critical_pair_error_total"] = df.get("pair_err_esophagitis_normal_z_line", 0).fillna(0) + df.get("pair_err_dyed_lifted_vs_resection", 0).fillna(0)
ranked_df = df.sort_values(["test_macro_f1", "critical_pair_error_total", "best_val_loss"], ascending=[False, True, True]).reset_index(drop=True)
display_cols = [c for c in ["run_name", "run_tag", "test_macro_f1", "best_val_acc", "best_val_loss", "pair_err_esophagitis_normal_z_line", "pair_err_dyed_lifted_vs_resection", "critical_pair_error_total", "learning_rate", "weight_decay"] if c in ranked_df.columns]
print(f"Toplam kayit: {len(ranked_df)}")
print("\nEn iyi 5 kosu:")
display(ranked_df[display_cols].head(5))

best_run = ranked_df.iloc[0]
print("\nSecilen en iyi kosu:")
for c in display_cols:
    print(f"{c}: {best_run[c]}")

print("\nYorum: once test_macro_f1 yuksek, sonra kritik ikili hata dusuk, sonra best_val_loss dusuk olacak sekilde siralandi.")
